# Xuất teacher targets: index 22 = layer thứ 23
Train và validation, 384 visual tokens, camera Front → Back, query cuối native prompt. Hai GPU, batch cố định 64/GPU, FP32. Không sinh đáp án hoặc training. Cache vision hiện có được đọc trực tiếp, không sửa/copy; không upload tự động.


Teacher loader: 2 spawned workers/GPU, persistent workers, pin memory, prefetch 1. Cache FP16; CPU/H2D/model/targets FP32. Train shuffle seed 42; validation sequential, each split sharded by order[rank::2]. Only stress uses longest prompts. One pool handles stress and extraction; no auto batch/precision reduction. New output is `smolvlm_teacher_index22_train_val_loader`; do not resume old serial runs.


In [ ]:
from pathlib import Path
import os, subprocess, sys
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
SOURCE = Path('/kaggle/working/Log_Metrics')
if not SOURCE.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'smolvlm-attention',
                    'https://github.com/huyvanzzz/Log_Metrics.git', str(SOURCE)], check=True)
else:
    branch = subprocess.check_output(['git', '-C', str(SOURCE), 'branch', '--show-current'], text=True).strip()
    origin = subprocess.check_output(['git', '-C', str(SOURCE), 'remote', 'get-url', 'origin'], text=True).strip()
    dirty = subprocess.check_output(['git', '-C', str(SOURCE), 'status', '--porcelain', '--untracked-files=no'], text=True).strip()
    if branch != 'smolvlm-attention' or origin.rstrip('/') != 'https://github.com/huyvanzzz/Log_Metrics.git' or dirty:
        raise RuntimeError('Source branch/origin/local changes differ; preserve them and choose another SOURCE.')
    subprocess.run(['git', '-C', str(SOURCE), 'pull', '--ff-only', 'origin', 'smolvlm-attention'], check=True)
os.chdir(SOURCE)
sys.path.insert(0, str(SOURCE))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(SOURCE / 'smolvlm_attention/sample_requirements.txt')], check=True)
print('Source commit:', subprocess.check_output(['git', '-C', str(SOURCE), 'rev-parse', 'HEAD'], text=True).strip())


## Chọn input và new/resume
`CACHE_INPUT`: thư mục Kaggle Input chứa cache hiện có; để trống để tự tìm đúng fingerprint hoặc gọi kagglehub. `PREPARED_DIR`: prepared data đã mount; để trống để tải public HF đã pin. Không cần publishing credentials.

`RESUME_INPUT`: thư mục đã giải nén partial resume ZIP, có run.json và internal/prompts, internal/shards. Chỉ copy artifacts này vào OUTPUT mới. Giữ nguyên code/dependencies và chọn MODE='resume'.

`COMPLETE_EXPORT_INPUT`: thư mục chứa train/ và val/ hoàn chỉnh; chỉ validate/đóng gói, không tải checkpoint/cache và không cần GPU. Để trống khi xuất mới.


In [ ]:
# Loader version: fresh output; old serial targets cannot resume here.
MODE = 'new'
OUTPUT = Path('/kaggle/working/smolvlm_teacher_index22_train_val_loader')
CACHE_INPUT = ''
PREPARED_DIR = ''
RESUME_INPUT = ''
COMPLETE_EXPORT_INPUT = ''
CONFIG = SOURCE / 'configs/smolvlm_teacher_index22_train_val.json'
BASE = [sys.executable, '-m', 'smolvlm_attention.teacher_targets']
def run(*args):
    command = ([sys.executable, '-m', 'smolvlm_attention.teacher_targets.diagnostics']
               if args and args[0] == 'run' else BASE)
    subprocess.run([*command, *map(str, args)], check=True, cwd=SOURCE)
if MODE not in ('new', 'resume'):
    raise ValueError('MODE must be new or resume')
if COMPLETE_EXPORT_INPUT and RESUME_INPUT:
    raise ValueError('Choose complete reuse or partial resume, not both')
COMPLETE_ROOT = Path(COMPLETE_EXPORT_INPUT) if COMPLETE_EXPORT_INPUT else None
if COMPLETE_ROOT is None and all((OUTPUT / 'export' / s / 'manifest.json').is_file() for s in ('train', 'val')):
    COMPLETE_ROOT = OUTPUT / 'export'
if COMPLETE_ROOT is not None:
    import json
    from smolvlm_attention.teacher_targets.runtime import load_config
    for split in ('train', 'val'):
        metadata = json.loads((COMPLETE_ROOT / split / 'manifest.json').read_text())
        if metadata.get('identity', {}).get('protocol') != load_config(CONFIG):
            raise ValueError('Complete export uses an older/different protocol; use the new loader output.')
    run('validate', '--export', COMPLETE_ROOT)
    print('Complete export validated; skip input preparation and GPU extraction.')
else:
    if RESUME_INPUT:
        import shutil
        source = Path(RESUME_INPUT).resolve()
        if MODE != 'resume':
            raise ValueError('RESUME_INPUT requires MODE=resume')
        if source != OUTPUT.resolve():
            if OUTPUT.exists() and any(OUTPUT.iterdir()):
                raise RuntimeError('Resume copy requires an empty/new OUTPUT; existing data will not be overwritten.')
            required = [source / 'run.json', source / 'internal/prompts', source / 'internal/shards', source / 'internal/schedule']
            if not all(p.exists() for p in required):
                raise FileNotFoundError('Partial resume needs run.json, internal/prompts, internal/shards and internal/schedule')
            if any(p.is_symlink() for base in required for p in ([base] + list(base.rglob('*')) if base.is_dir() else [base])):
                raise ValueError('Resume artifacts must not contain symbolic links')
            OUTPUT.mkdir(parents=True, exist_ok=True)
            shutil.copy2(source / 'run.json', OUTPUT / 'run.json')
            for name in ('prompts', 'shards', 'schedule'):
                shutil.copytree(source / 'internal' / name, OUTPUT / 'internal' / name)
    import torch, transformers, tokenizers
    assert transformers.__version__ == '4.46.3' and tokenizers.__version__ == '0.20.4'
    if torch.cuda.device_count() < 2:
        raise RuntimeError('Teacher extraction requires two CUDA GPUs; complete-export reuse does not.')
    print({'torch': torch.__version__, 'gpus': [torch.cuda.get_device_name(i) for i in (0, 1)],
           'precision': 'FP32, autocast/TF32 off in workers', 'batch_per_gpu': 64})


## Validate inputs và tạo compact prompt mapping
Parent quét cache một lần; workers kiểm tra receipt và từng batch. Prompt không chứa answer. Index 22 là layer thứ 23. Bước này tokenizes train/val và báo counts/identity/dung lượng. Không cần chạy lại cell khi đã thành công trong cùng phiên.


In [ ]:
if COMPLETE_ROOT is None:
    args = ['prepare', '--output', str(OUTPUT), '--config', str(CONFIG), '--mode', MODE]
    if CACHE_INPUT:
        args += ['--cache-input', CACHE_INPUT]
    if PREPARED_DIR:
        args += ['--prepared-dir', PREPARED_DIR]
    run(*args)
    RUN_PATH = OUTPUT / 'run.json'


## Stress hai GPU rồi xuất targets
Mỗi GPU kiểm tra 64 prompt dài nhất của từng shard. Cả hai phải pass mới bắt đầu ghi. OOM/lỗi thì dừng cả hai, không giảm batch hoặc bỏ mẫu. Logs theo phase/batch, không in từng mẫu; logs chi tiết ở OUTPUT/logs/gpu_0.log và gpu_1.log.

Nếu ngắt cell, chờ worker dừng hẳn rồi mới đóng gói partial. Lần sau chọn MODE='resume'. Không chạy đồng thời hai notebook ghi cùng OUTPUT.


In [ ]:
if COMPLETE_ROOT is None:
    run('run', '--run', RUN_PATH)
    COMPLETE_ROOT = OUTPUT / 'export'


## Validate và đóng gói complete portable export
ZIP chỉ chứa targets/mapping/manifest; không có cache vision, checkpoint, raw images, prepared data, answers hoặc full prompt IDs. Numeric payload hai split khoảng 0,517 GiB. ZIP dùng stored và không materialize toàn tensor vào RAM.


In [ ]:
run('validate', '--export', COMPLETE_ROOT)
if (OUTPUT / 'run.json').is_file() and COMPLETE_ROOT.resolve() == (OUTPUT / 'export').resolve():
    run('bundle', '--run', OUTPUT / 'run.json')
    ARCHIVE = OUTPUT / 'teacher_index22_train_val_complete.zip'
else:
    import zipfile
    from smolvlm_attention.sample_store import atomic_json, digest
    OUTPUT.mkdir(parents=True, exist_ok=True)
    ARCHIVE = OUTPUT / 'teacher_index22_train_val_complete.zip'
    temporary = Path(str(ARCHIVE) + '.tmp')
    with zipfile.ZipFile(temporary, 'w', compression=zipfile.ZIP_STORED, allowZip64=True) as archive:
        for split in ('train', 'val'):
            for name in ('log_probs.npy', 'samples.jsonl', 'manifest.json'):
                archive.write(COMPLETE_ROOT / split / name, f'export/{split}/{name}')
    os.replace(temporary, ARCHIVE)
    atomic_json(ARCHIVE.with_suffix('.sha256.json'), {'filename': ARCHIVE.name, 'sha256': digest(ARCHIVE)})
from IPython.display import FileLink, display
print('Complete archive:', ARCHIVE)
display(FileLink(str(ARCHIVE)))


## Chỉ khi chưa hoàn tất: đóng gói partial để resume
Cell mặc định tắt. Chỉ bật sau khi extraction đã dừng; lock từ chối đóng gói khi worker còn ghi. Partial ZIP chứa shards, commit records, compact prompt mapping và run metadata. Không dùng partial làm teacher dataset hoàn chỉnh. Giải nén ở phiên sau, điền RESUME_INPUT và MODE='resume'.


In [ ]:
BUNDLE_PARTIAL = False
if BUNDLE_PARTIAL:
    run('bundle', '--run', OUTPUT / 'run.json', '--partial')
    from IPython.display import FileLink, display
    display(FileLink(str(OUTPUT / 'teacher_index22_train_val_partial_resume.zip')))
